In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Transpilacja z menedżerami przejść

<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Zalecanym sposobem transpilacji Circuit-u jest utworzenie etapowego menedżera przejść (staged pass manager), a następnie wywołanie jego metody `run` z Circuit-em jako argumentem. Ta strona wyjaśnia, jak transpilować obwody kwantowe w ten sposób.
## Co to jest (etapowy) menedżer przejść?
W kontekście Qiskit SDK transpilacja oznacza proces przekształcania wejściowego Circuit-u do postaci nadającej się do wykonania na urządzeniu kwantowym. Transpilacja odbywa się zazwyczaj w sekwencji kroków zwanych przejściami Transpilera (transpiler passes). Circuit jest przetwarzany przez każde przejście Transpilera kolejno, a wyjście jednego przejścia staje się wejściem następnego. Na przykład jedno przejście może przejrzeć Circuit i scalić wszystkie kolejne sekwencje bramek jednoQubitowych, a następne przejście może zsyntetyzować te Gate-y w zbiorze bazowym docelowego urządzenia. Przejścia Transpilera dostarczone z Qiskit znajdują się w module [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Menedżer przejść (pass manager) to obiekt przechowujący listę przejść Transpilera i mogący je wykonywać na Circuit-ach. Utwórz menedżer przejść, inicjując obiekt [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) z listą przejść Transpilera. Aby uruchomić transpilację na Circuit-cie, wywołaj metodę [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) z Circuit-em jako argumentem.

Etapowy menedżer przejść (staged pass manager) to szczególny rodzaj menedżera przejść reprezentujący wyższy poziom abstrakcji niż zwykły menedżer przejść. Podczas gdy zwykły menedżer przejść składa się z kilku przejść Transpilera, etapowy menedżer przejść składa się z kilku *menedżerów przejść*. Jest to przydatna abstrakcja, ponieważ transpilacja odbywa się zazwyczaj w odrębnych etapach, opisanych w [Etapach Transpilera](transpiler-stages), przy czym każdy etap reprezentowany jest przez menedżer przejść. Etapowe menedżery przejść są reprezentowane przez klasę [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). Pozostała część tej strony opisuje, jak tworzyć i dostosowywać (etapowe) menedżery przejść.
## Generowanie wstępnie zdefiniowanego etapowego menedżera przejść
Aby utworzyć wstępnie zdefiniowany etapowy menedżer przejść z rozsądnymi ustawieniami domyślnymi, użyj funkcji [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager).

> **Note:** W tych przykładach używany jest fikcyjny Backend `FakeSherbrooke` z `qiskit_ibm_runtime`, ale możesz wypróbować go na dowolnym rzeczywistym lub fikcyjnym Backendzie zgodnym z Qiskit. Twoje wyniki mogą być inne.

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Aby transpilować Circuit lub listę Circuit-ów za pomocą menedżera przejść, przekaż Circuit lub listę Circuit-ów do metody `run`. Zróbmy to na dwuQubitowym Circuit-cie składającym się z bramki Hadamarda, po której następują dwie sąsiednie bramki CX:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg)

Opis możliwych argumentów funkcji `generate_preset_pass_manager` znajdziesz w [Domyślnych ustawieniach transpilacji i opcjach konfiguracji](defaults-and-configuration-options). Argumenty funkcji `generate_preset_pass_manager` odpowiadają argumentom funkcji [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Masz trudności z zapamiętaniem szczegółów menedżera przejść? Spróbuj zapytać Qiskit Code Assistant." />

Jeśli wstępnie zdefiniowane menedżery przejść nie spełniają twoich potrzeb, dostosuj transpilację, tworząc własne (etapowe) menedżery przejść, a nawet własne przejścia transpilacji. Pozostała część tej strony opisuje, jak tworzyć menedżery przejść. Instrukcje dotyczące tworzenia własnych przejść transpilacji znajdziesz w [Napisz własne przejście Transpilera](custom-transpiler-pass).

## Utwórz własny menedżer przejść

Moduł [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) zawiera wiele przejść Transpilera, których można użyć do tworzenia menedżerów przejść. Aby utworzyć menedżer przejść, zainicjuj obiekt `PassManager` z listą przejść. Na przykład poniższy kod tworzy przejście Transpilera, które scala sąsiednie dwuQubitowe Gate-y, a następnie syntetyzuje je w bazie bramek [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) i [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Aby zademonstrować ten menedżer przejść w działaniu, przetestuj go na dwuQubitowym Circuit-cie składającym się z bramki Hadamarda, po której następują dwie sąsiednie Gate-y CX:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg)

Aby uruchomić menedżer przejść na Circuit-cie, wywołaj metodę `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg)

Bardziej zaawansowany przykład pokazujący, jak utworzyć menedżer przejść w celu implementacji techniki tłumienia błędów zwanej dynamicznym odsprzęganiem (dynamical decoupling), znajdziesz w [Utwórz menedżer przejść dla dynamicznego odsprzęgania](dynamical-decoupling-pass-manager).

## Utwórz etapowy menedżer przejść

`StagedPassManager` to menedżer przejść składający się z poszczególnych etapów, przy czym każdy etap jest zdefiniowany przez instancję `PassManager`. Możesz utworzyć `StagedPassManager`, określając żądane etapy. Na przykład poniższy kod tworzy etapowy menedżer przejść z dwoma etapami: `init` i `translation`. Etap `translation` jest zdefiniowany przez menedżer przejść utworzony wcześniej.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Nie ma ograniczeń co do liczby etapów w etapowym menedżerze przejść.

Innym przydatnym sposobem tworzenia etapowego menedżera przejść jest rozpoczęcie od wstępnie zdefiniowanego etapowego menedżera przejść, a następnie wymienienie niektórych etapów. Na przykład poniższy kod generuje wstępnie zdefiniowany menedżer przejść z poziomem optymalizacji 3, a następnie określa własny etap `pre_layout`.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[Funkcje generatorów etapów](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) mogą być przydatne do budowania własnych menedżerów przejść.
Generują etapy zapewniające typową funkcjonalność używaną w wielu menedżerach przejść.
Na przykład [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) może być użyta do wygenerowania etapu „osadzania" wybranego początkowego `Layout` z przejścia układu w określonym urządzeniu docelowym.

## Kolejne kroki
> **Tip:** - [Napisz własne przejście Transpilera](custom-transpiler-pass).
> - [Utwórz menedżer przejść dla dynamicznego odsprzęgania](dynamical-decoupling-pass-manager).
> - Aby dowiedzieć się więcej o funkcji `generate_preset_passmanager`, zapoznaj się z tematem [Domyślne ustawienia transpilacji i opcje konfiguracji](defaults-and-configuration-options).
> - Wypróbuj przewodnik [Porównaj ustawienia Transpilera](/guides/circuit-transpilation-settings).
> - Zapoznaj się z [dokumentacją API Transpilera](https://docs.quantum.ibm.com/api/qiskit/transpiler).